In [1]:
import pandas as pd
import numpy as np

# Cố định random seed để mỗi lần chạy đều ra kết quả giống nhau
np.random.seed(42)

# Số lượng khách hàng giả lập
n_customers = 5000

# Khởi tạo DataFrame trống
df = pd.DataFrame()
df['Customer_ID'] = [f'CUST_{str(i).zfill(5)}' for i in range(1, n_customers + 1)]

Sinh dữ liệu Nhân khẩu học (Demographics)
Chúng ta sẽ dùng phân phối chuẩn (Normal Distribution) và random choice để tạo độ tuổi và khu vực sống.

In [2]:
# 1. Tuổi: Trung bình 28 tuổi, lệch chuẩn 8. Giới hạn từ 16 đến 60.
df['Customer_Age'] = np.random.normal(loc=28, scale=8, size=n_customers).astype(int)
df['Customer_Age'] = np.clip(df['Customer_Age'], 16, 60)

# 2. Khu vực sống: Tier 1 (Thành phố lớn - 40%), Tier 2 (Tỉnh lẻ - 60%)
df['Location_Type'] = np.random.choice(['Tier 1', 'Tier 2'], size=n_customers, p=[0.4, 0.6])

# 3. Số tháng là thành viên: Từ 1 tháng đến 5 năm (60 tháng)
df['Membership_Months'] = np.random.randint(1, 61, size=n_customers)

Sinh dữ liệu Giao dịch cốt lõi (RFM & Brands)
Đây là dữ liệu nền tảng. Chúng ta sẽ làm cho nó logic một chút (ví dụ: Khách Tier 1 thường chi tiêu cao hơn).

In [3]:
# 1. Tần suất mua hàng (Frequency): Đa số mua 1-3 lần, rất ít người mua > 10 lần (Phân phối Poisson)
df['Frequency'] = np.random.poisson(lam=2.5, size=n_customers)
df['Frequency'] = np.clip(df['Frequency'], 1, 30) # Cắt tối thiểu là 1

# 2. Độ gần đây (Recency): Từ 1 đến 365 ngày
df['Recency'] = np.random.randint(1, 365, size=n_customers)

# 3. Thương hiệu yêu thích
brands = ['Apple', 'Samsung', 'Oppo', 'Xiaomi']
df['Favorite_Brand'] = np.random.choice(brands, size=n_customers, p=[0.35, 0.3, 0.2, 0.15])

# 4. Sức mua (Monetary): Logic = (Tần suất * Trung bình 1 đơn). 
# Fan Apple và người ở Tier 1 thường có đơn giá cao hơn.
base_order_value = np.where(df['Favorite_Brand'] == 'Apple', 15000000, 8000000)
base_order_value = np.where(df['Location_Type'] == 'Tier 1', base_order_value * 1.2, base_order_value)

# Thêm độ nhiễu (noise) để dữ liệu tự nhiên
noise = np.random.normal(1, 0.2, n_customers) 
df['Monetary'] = (df['Frequency'] * base_order_value * noise).astype(int)
df['Monetary'] = np.round(df['Monetary'], -4) # Làm tròn tới hàng chục nghìn

Sinh các tỷ lệ hành vi đặc thù (Các biến mới)
Đây là phần "ăn điểm" mà chúng ta đã thảo luận.

In [4]:
# Các tỷ lệ từ 0 -> 1
df['Installment_Rate'] = np.random.uniform(0, 1, n_customers)
df['Flagship_Ratio'] = np.random.uniform(0, 1, n_customers)
df['Accessories_Ratio'] = np.random.uniform(0, 1, n_customers)
df['Credit_Card_Usage'] = np.random.uniform(0, 1, n_customers)

# Số lần đi bảo hành (Warranty Claims): Đa số là 0, rất ít là 1 hoặc 2
df['Warranty_Claims'] = np.random.choice([0, 1, 2, 3], size=n_customers, p=[0.7, 0.2, 0.08, 0.02])

# Số lần mở App trong 30 ngày qua
df['App_Logins_L30D'] = np.random.randint(0, 50, size=n_customers)

Định nghĩa nhãn VIP (Gán Label cho Machine Learning)
Để mô hình XGBoost của bạn học được, dữ liệu phải có tính quy luật. Bạn phải đóng vai "Giám đốc kinh doanh" để thiết lập luật: Thế nào là VIP?

In [5]:
# KHỞI TẠO LUẬT KINH DOANH (BUSINESS RULES) ĐỂ XÁC ĐỊNH VIP
# Một khách hàng được coi là VIP thực sự nếu thỏa mãn các điều kiện KHÓ:
condition_1 = df['Monetary'] > 30000000         # Chi tiêu trên 30 triệu
condition_2 = df['Frequency'] >= 3              # Mua từ 3 lần trở lên
condition_3 = df['Warranty_Claims'] <= 1        # Ít phàn nàn/bảo hành
condition_4 = df['Flagship_Ratio'] > 0.4        # Hay mua hàng xịn
condition_5 = df['Recency'] < 180               # Còn tương tác trong 6 tháng qua

# Tính điểm VIP Score (Càng thỏa mãn nhiều điều kiện thì điểm càng cao)
df['VIP_Score'] = condition_1.astype(int) + condition_2.astype(int) + condition_3.astype(int) + condition_4.astype(int) + condition_5.astype(int)

# Gán nhãn: Nếu thỏa mãn từ 4/5 điều kiện trở lên -> VIP (1), còn lại là Thường (0)
df['Is_VIP'] = np.where(df['VIP_Score'] >= 4, 1, 0)

# Xóa cột VIP_Score đi để AI tự tìm ra quy luật thay vì nhìn vào đáp án
df = df.drop(columns=['VIP_Score'])

# Kiểm tra xem tỷ lệ VIP / Thường có cân bằng không
print("Tỷ lệ phân bổ khách hàng:")
print(df['Is_VIP'].value_counts(normalize=True) * 100)

Tỷ lệ phân bổ khách hàng:
Is_VIP
0    70.24
1    29.76
Name: proportion, dtype: float64


Lưu ra file CSV

In [7]:
import os

# Tạo thư mục data nếu chưa có
os.makedirs('../data/processed', exist_ok=True)

# Lưu tệp
file_path = '../data/processed/real_world_mobile_data.csv'
df.to_csv(file_path, index=False)
print(f"🎉 Đã tạo thành công tệp dữ liệu giả lập tại: {file_path}")

🎉 Đã tạo thành công tệp dữ liệu giả lập tại: ../data/processed/real_world_mobile_data.csv
